# Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Data processing and math
import pandas as pd
import numpy as np

# Statistics
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display handling
import warnings


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


##########
# LABELS #
##########
labels_measure = {
    'measure_action': 'Responsibility for Action'
}


#################
# VISUALIZATION #
#################
palette_main = {
    "Good":     "#0072B2", "Bad":   "#D55E00",
    "Activist": "#56B4E9", "Bigot": "#E69F00",
    "Help":     "#F0E442", "Harm":  "#882255"
}

# Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_pilot_1_data.csv')

# Define factors
factors_map = {
     5: {'Upbringing': "Bad",  'Belief': "Activist", 'Action': "Help"},
     6: {'Upbringing': "Bad",  'Belief': "Activist", 'Action': "Harm"},
     7: {'Upbringing': "Bad",  'Belief': "Bigot",    'Action': "Help"},
     8: {'Upbringing': "Bad",  'Belief': "Bigot",    'Action': "Harm"},
    13: {'Upbringing': "Good", 'Belief': "Activist", 'Action': "Help"},
    14: {'Upbringing': "Good", 'Belief': "Activist", 'Action': "Harm"},
    15: {'Upbringing': "Good", 'Belief': "Bigot",    'Action': "Help"},
    16: {'Upbringing': "Good", 'Belief': "Bigot",    'Action': "Harm"}
}
    
# Reshape wide to long
indices = [5, 6, 7, 8, 13, 14, 15, 16]
long_data = []

for _, row in df.iterrows():
    p_id = row['ID']
    
    for i in indices:
        col_action = f'homophobia.{i}-pAction'
        
        if col_action in df.columns and pd.notna(row[col_action]):
            score_action = pd.to_numeric(row[col_action], errors = 'coerce') - 7
            
            entry = {
                'ID': p_id,
                'measure_action': score_action
            }
            entry.update(factors_map[i])
            
            long_data.append(entry)

df_long = pd.DataFrame(long_data)

print(f"Transformation complete ({len(df_long)} Observations).")

# Calculate Descriptive Statistics

In [ ]:
# Calculate descriptive statistics
descriptive_stats = df_long.groupby(['Upbringing', 'Belief', 'Action'])['measure_action'].agg(['mean', 'std', 'count']).round(3)

# Display results
display(descriptive_stats)

# Perform ANOVA

In [ ]:
# Define formula
formula = "measure_action ~ C(Upbringing) * C(Belief) * C(Action)"
model = ols(formula, data = df_long).fit()

# Run ANOVA
aov_table = sm.stats.anova_lm(model, typ = 2)

# Calculate effect sizes
aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])

# Display results
display(aov_table.round(3))

# Generate Bar Plot

In [ ]:
# Create graph
g = sns.catplot(data = df_long, x = "Upbringing", y = "measure_action",
                hue = "Belief", col = "Action", kind = "bar",
                palette = palette_main, capsize = .05, height = 5)

# Set axis labels and titles
g.set_axis_labels("Upbringing", "Mean Score (–6 to +6)")
g.set_titles("{col_name}")

# Add horizontal zero lines
for ax in g.axes.flat:
    ax.axhline(0, color = 'black', lw = 1)
    ax.set_ylim(-6, 6)

# Add main title
plt.subplots_adjust(top = 0.85)
g.fig.suptitle('Responsibility for Action', fontsize = 16)

# Export graph
filename = 'blame_praise_self_pilot_1_bar_plot.png'
g.savefig(filename, dpi = 300, bbox_inches = 'tight')
plt.show()
print(f"Graph saved as '{filename}'.")